End-to-End Machine Learning Classification: Diabetes Prediction

Download guide: go to https://www.kaggle.com/datasets/iammustafatz/diabetes-prediction-dataset, click on download, unzip the downloaded archive in the same folder as the notebook

I. Dataset Description
Source of the dataset:
Kaggle - Diabetes Prediction Dataset by Mustafa Ali(Link: https://www.kaggle.com/datasets/iammustafatz/diabetes-prediction-dataset).
Problem Context & Classification Target: Diabetes is a chronic metabolic disease that affects millions worldwide. Early detection is critical for managing the condition and preventing severe complications. The real-world problem this dataset tackles is the early identification of patients at risk of diabetes based on their demographic information and medical history.
Target Variable: diabetes (Binary: 0 indicates the absence of diabetes, 1 indicates the presence of diabetes). 

Feature Descriptions:
1. gender (Categorical): Biological gender of the patient (Female, Male, Other). Gender can be relevant as certain physiological differences may impact diabetes risk.  
2. age (Numeric - Float): The age of the patient in years. Age is a major risk factor; type 2 diabetes risk increases significantly as people get older.  
3. hypertension (Binary): Indicates if the patient has high blood pressure (0 = No, 1 = Yes). Hypertension is a common comorbidity with diabetes.  
4. heart_disease (Binary): Indicates if the patient has cardiovascular disease (0 = No, 1 = Yes). Heart conditions are heavily correlated with metabolic disorders.  
5. smoking_history (Categorical): The patient's smoking status (e.g., current, former, never, No Info). Smoking increases oxidative stress and inflammation, which are linked to diabetes risk.  
6. bmi (Numeric - Float): Body Mass Index, a measure of body fat based on weight and height. A higher BMI (overweight/obesity) is one of the leading risk factors for type 2 diabetes.  
7. HbA1c_level (Numeric - Float): Hemoglobin A1c level, representing the average blood sugar level over the past 2-3 months. Levels above 6.5% strongly indicate diabetes.  
8. blood_glucose_level (Numeric - Integer): Current amount of glucose in the bloodstream (mg/dL). Consistently high glucose is the primary indicator of diabetes.  

Dataset Statistics: 
-Instances: 100,000 patient records.  
-Features: 8 predictive features + 1 target variable.  
-Class Distribution: The dataset is imbalanced. Approximately 91.5% of instances are negative (Class 0: No diabetes) and 8.5% are positive (Class 1: Diabetes). 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier


csv_path = "diabetes_prediction_dataset.csv"

df = pd.read_csv(csv_path)

df = df.drop_duplicates()

print(f"Dataset shape: {df.shape}")
print("\nClass Distribution:")
print(df['diabetes'].value_counts(normalize=True) * 100)
df.head()

Dataset shape: (96146, 9)

Class Distribution:
diabetes
0    91.178
1     8.822
Name: proportion, dtype: float64


,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes
0,Female,80.0,0,1,never,25.19,6.6,140,0
1,Female,54.0,0,0,No Info,27.32,6.6,80,0
2,Male,28.0,0,0,never,27.32,5.7,158,0
3,Female,36.0,0,0,current,23.45,5.0,155,0
4,Male,76.0,1,1,current,20.14,4.8,155,0


II. Data Preprocessing, Scaling, and Encoding
(Goal Justification):
Machine learning models require numerical input. We must encode categorical variables (gender, smoking_history). We will use OrdinalEncoder for smoking history since there is a loose progression (never -> former -> current), and one-hot encoding for gender. Scaling is absolutely necessary for distance-based models or models that rely on variance, but it also helps algorithms converge faster. We will apply StandardScaler to numeric features (age, bmi, HbA1c_level, blood_glucose_level).

In [34]:
# 1. Encoding
# One-Hot Encoding for Gender
df_processed = pd.get_dummies(df, columns=['gender'], drop_first=True)

# Ordinal Encoding for Smoking History
smoking_order = ['No Info', 'never', 'not current', 'former', 'ever', 'current']
encoder = OrdinalEncoder(categories=[smoking_order])
df_processed['smoking_history'] = encoder.fit_transform(df_processed[['smoking_history']])

# 2. Scaling
scaler = StandardScaler()
numeric_cols = ['age', 'bmi', 'HbA1c_level', 'blood_glucose_level']
df_processed[numeric_cols] = scaler.fit_transform(df_processed[numeric_cols])

X = df_processed.drop('diabetes', axis=1)
y = df_processed['diabetes']

X.head()

,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,gender_Male,gender_Other
0,1.700840,0,1,1.0,-0.314947,0.994563,0.043554,False,False
1,0.543372,0,0,0.0,-0.000216,0.994563,-1.423096,False,False
2,-0.614096,0,0,1.0,-0.000216,0.155970,0.483549,True,False
3,-0.257952,0,0,5.0,-0.572051,-0.496269,0.410216,False,False
4,1.522768,1,1,5.0,-1.061141,-0.682623,0.410216,True,False


III. Train/Test Split & Stratification
(Goal Justification):
We split the data into training (80%) and testing (20%) sets to evaluate how well our models generalize to unseen data. Because our dataset has a severe class imbalance (~91% negative vs ~9% positive), stratification is strictly necessary. Setting stratify=y guarantees that the 91/9 ratio is preserved in both the training and testing sets, preventing the models from training on a disproportionate amount of majority class examples.

In [35]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} instances")
print(f"Testing set: {X_test.shape[0]} instances")

Training set: 76916 instances
Testing set: 19230 instances


IV. Performance Metric Selection
(Goal Justification):
Because of the heavy class imbalance, Accuracy is a misleading metric. A dummy model predicting "0" for everyone would achieve 91% accuracy but fail completely at identifying diabetics.

We will primarily use the F1-Score (the harmonic mean of Precision and Recall) because it balances the trade-off between false positives and false negatives.

Recall is also highly critical here: in a medical context, it is usually better to have a False Positive (telling a healthy person they might have diabetes and doing further tests) than a False Negative (sending a diabetic patient home untreated).

V. Feature Selection
(Goal Justification):
Feature selection reduces dimensionality, which speeds up training, creates "slimmer" models, and reduces the risk of overfitting to noise. We will use SelectKBest with ANOVA F-value (f_classif) to select the top 6 most informative features out of the 9 available dummy/encoded variables.

In [36]:
selector = SelectKBest(score_func=f_classif, k=6)
X_train_selected = selector.fit_transform(X_train, y_train)
X_test_selected = selector.transform(X_test)

selected_features = X.columns[selector.get_support()]
print("Selected Features for slim models:", list(selected_features))

Selected Features for slim models: ['age', 'hypertension', 'heart_disease', 'bmi', 'HbA1c_level', 'blood_glucose_level']


VI. Model Selection, Optimization, and Evaluation(Goal Justification):
Decision Tree: 
    A highly interpretable model (crucial for medicine), but prone to overfitting. We will optimize it using RandomizedSearchCV (tuning max depth and min samples).
Naive Bayes (GaussianNB):
    We chose this over SVM because SVM scales quadratically with the number of samples ($O(n^2)$). With nearly 100,000 instances, SVM would take an unfeasibly long time to train without aggressive downsampling. Naive Bayes is extremely fast and handles large datasets well.
Random Forest (Ensemble):
    Builds multiple decision trees to reduce variance and prevent the overfitting seen in single Decision Trees.To avoid overfitting/underfitting, we use Cross-Validation during our Randomized Search to find the sweet spot for hyperparameters.

In [ ]:
def evaluate_model(model, X_tr, y_tr, X_te, y_te, model_name):
    y_pred_tr = model.predict(X_tr)
    y_pred_te = model.predict(X_te)
    
    print(f"--- {model_name} ---")
    print(f"Train F1: {f1_score(y_tr, y_pred_tr):.4f} | Test F1: {f1_score(y_te, y_pred_te):.4f}")
    print(f"Test Recall: {recall_score(y_te, y_pred_te):.4f}")
    print("Classification Report (Test):\n", classification_report(y_te, y_pred_te))
    print("-" * 30 + "\n")

# 1. Decision Tree 
dt_params = {
    'max_depth': [5, 10, 15, 20, None],
    'min_samples_split': [2, 10, 20],
    'min_samples_leaf': [1, 5, 10],
    'class_weight': ['balanced', None]
}
dt = DecisionTreeClassifier(random_state=42)
dt_search = RandomizedSearchCV(dt, dt_params, n_iter=10, scoring='f1', cv=3, random_state=42, n_jobs=-1)
dt_search.fit(X_train_selected, y_train)

evaluate_model(dt_search.best_estimator_, X_train_selected, y_train, X_test_selected, y_test, "Decision Tree (Optimized)")

# 2. Naive Bayes
nb = GaussianNB()
nb.fit(X_train_selected, y_train)
evaluate_model(nb, X_train_selected, y_train, X_test_selected, y_test, "Gaussian Naive Bayes")

# 3. Random Forest 
rf_params = {
    'n_estimators': [50, 100],
    'max_depth': [10, 15, 20],
    'class_weight': ['balanced', 'balanced_subsample']
}
rf = RandomForestClassifier(random_state=42)
rf_search = RandomizedSearchCV(rf, rf_params, n_iter=5, scoring='f1', cv=3, random_state=42, n_jobs=-1)
rf_search.fit(X_train_selected, y_train)

evaluate_model(rf_search.best_estimator_, X_train_selected, y_train, X_test_selected, y_test, "Random Forest (Optimized)")

--- Decision Tree (Optimized) ---
Train F1: 0.8087 | Test F1: 0.8076
Test Recall: 0.6893
Classification Report (Test):
               precision    recall  f1-score   support

           0       0.97      1.00      0.98     17534
           1       0.97      0.69      0.81      1696

    accuracy                           0.97     19230
   macro avg       0.97      0.84      0.90     19230
weighted avg       0.97      0.97      0.97     19230

------------------------------

--- Gaussian Naive Bayes ---
Train F1: 0.5394 | Test F1: 0.5366
Test Recall: 0.6468
Classification Report (Test):
               precision    recall  f1-score   support

           0       0.96      0.93      0.94     17534
           1       0.46      0.65      0.54      1696

    accuracy                           0.90     19230
   macro avg       0.71      0.79      0.74     19230
weighted avg       0.92      0.90      0.91     19230

------------------------------

--- Random Forest (Optimized) ---
Train F1: 0.9

VII. Proposing New Features (Feature Engineering)
(Goal Justification):
Domain knowledge in medicine tells us that combining certain features can create strong predictors.

BMI_Age_Interaction: The risk of metabolic issues compounds as people get older and heavier. Multiplying age by BMI captures this synergistic risk.

Clinical_Risk_Score: A simple sum of clinical red flags (Hypertension + Heart Disease).

Pre_Diabetes_Flag: High glucose and high HbA1c are the direct measures of diabetes. We can create a boolean flag if their unscaled values are in the pre-diabetic ranges (HbA1c > 6.0 and Glucose > 125).

We will create these features on the raw dataset, re-scale, re-split, and re-evaluate our best model (Random Fores
t) to see if performance improves.

In [ ]:
df_new = df.copy()

# 1. Interaction Feature
df_new['bmi_age_interaction'] = df_new['bmi'] * df_new['age']

# 2. Cumulative Clinical Risk
df_new['clinical_risk'] = df_new['hypertension'] + df_new['heart_disease']

# 3. Pre-diabetes logical flag 
df_new['pre_diabetes_flag'] = ((df_new['HbA1c_level'] > 6.0) | (df_new['blood_glucose_level'] > 125)).astype(int)

# Re-encode and Re-scale
df_new = pd.get_dummies(df_new, columns=['gender'], drop_first=True)
df_new['smoking_history'] = encoder.fit_transform(df_new[['smoking_history']])

numeric_cols_new = ['age', 'bmi', 'HbA1c_level', 'blood_glucose_level', 'bmi_age_interaction']
df_new[numeric_cols_new] = scaler.fit_transform(df_new[numeric_cols_new])

# Re-Split
X_new = df_new.drop('diabetes', axis=1)
y_new = df_new['diabetes']

X_tr_new, X_te_new, y_tr_new, y_te_new = train_test_split(
    X_new, y_new, test_size=0.2, random_state=42, stratify=y_new
)

# Feature Selection 
selector_new = SelectKBest(score_func=f_classif, k=8)
X_tr_new_sel = selector_new.fit_transform(X_tr_new, y_tr_new)
X_te_new_sel = selector_new.transform(X_te_new)

print("Newly Selected Features:", list(X_new.columns[selector_new.get_support()]))

# Train and Evaluate Random Forest
rf_new = RandomForestClassifier(n_estimators=100, max_depth=15, class_weight='balanced', random_state=42)
rf_new.fit(X_tr_new_sel, y_tr_new)

print("\n--- Evaluation WITH Proposed Features ---")
evaluate_model(rf_new, X_tr_new_sel, y_tr_new, X_te_new_sel, y_te_new, "Random Forest (New Features)")

Newly Selected Features: ['age', 'hypertension', 'heart_disease', 'bmi', 'HbA1c_level', 'blood_glucose_level', 'bmi_age_interaction', 'clinical_risk']

--- Evaluation WITH Proposed Features ---
--- Random Forest (New Features) ---
Train F1: 0.7412 | Test F1: 0.6721
Test Recall: 0.8502
Classification Report (Test):
               precision    recall  f1-score   support

           0       0.98      0.93      0.96     17534
           1       0.56      0.85      0.67      1696

    accuracy                           0.93     19230
   macro avg       0.77      0.89      0.82     19230
weighted avg       0.95      0.93      0.93     19230

------------------------------

